# Etapa 3 — Probabilidade nas escolas do Amazonas
Censo Escolar 2025. Sorteamos, com chances iguais, uma escola entre as linhas do CSV da Etapa 1. Cada probabilidade é a frequência do evento nesse conjunto.

Usamos três indicadores binários (1 = sim, 0 = não). **L** indica laboratório de informática, **B** indica biblioteca e **C** é a classe `infra_digital_basica`. Na Etapa 1, C vale 1 quando a escola informa internet para alunos e ao menos um tipo de aparelho para eles (desktop, portátil ou tablet).

In [1]:
import pandas as pd

dados = pd.read_csv("dados_etapa1.csv")
colunas = ["IN_LABORATORIO_INFORMATICA", "IN_BIBLIOTECA", "infra_digital_basica"]
assert dados["CO_ENTIDADE"].is_unique
assert dados[colunas].isin([0, 1]).all().all()
L, B, C = (dados[coluna] == 1 for coluna in colunas)
N = len(dados)
print(f"Escolas analisadas: {N}")

Escolas analisadas: 5562


## 1. Distribuição conjunta
Cada escola pertence a exatamente uma das quatro combinações de laboratório e biblioteca. Dividimos a contagem de cada combinação pelo total de escolas.

In [2]:
contagens = pd.crosstab(L.astype(int), B.astype(int))
contagens.index.name = "L"
contagens.columns.name = "B"
conjunta = contagens / N
print("Número de escolas por evento")
print(contagens)
print("\nProbabilidade por evento")
print(conjunta.round(5))
print("Soma:", conjunta.to_numpy().sum().round(10))

Número de escolas por evento
B     0    1
L           
0  3994  613
1   213  742

Probabilidade por evento
B        0        1
L                  
0  0.71809  0.11021
1  0.03830  0.13341
Soma: 1.0


## 2. Marginais e condicionais
Somar uma linha da tabela dá a marginal de L. Somar uma coluna dá a de B. Na condicional, o denominador passa a contar apenas as escolas que satisfazem a condição.

In [3]:
print("Marginais de L:", conjunta.sum(axis=1).round(5).to_dict())
print("Marginais de B:", conjunta.sum(axis=0).round(5).to_dict())
print(f"P(B=1 | L=1) = {contagens.loc[1, 1]}/{L.sum()} = {B[L].mean():.2%}")
print(f"P(L=1 | B=1) = {contagens.loc[1, 1]}/{B.sum()} = {L[B].mean():.2%}")

Marginais de L: {0: 0.8283, 1: 0.1717}
Marginais de B: {0: 0.75638, 1: 0.24362}
P(B=1 | L=1) = 742/955 = 77.70%
P(L=1 | B=1) = 742/1355 = 54.76%


As duas condicionais contam as mesmas 742 escolas com ambos os recursos. A primeira toma como referência as 955 escolas com laboratório. A segunda usa as 1.355 com biblioteca.

## 3. Regra da cadeia
Para o evento em que os três indicadores valem 1, multiplicamos P(C) × P(L | C) × P(B | L, C). A cada fator restringimos o grupo usado no denominador.

In [4]:
p_c = C.mean()
p_l_c = L[C].mean()
p_b_lc = B[L & C].mean()
cadeia = p_c * p_l_c * p_b_lc
print(f"({C.sum()}/{N}) × ({(L & C).sum()}/{C.sum()}) × ({(B & L & C).sum()}/{(L & C).sum()}) = {cadeia:.4%}")
print(f"Contagem direta: {(B & L & C).sum()}/{N} = {(B & L & C).mean():.4%}")

(1007/5562) × (638/1007) × (496/638) = 8.9177%
Contagem direta: 496/5562 = 8.9177%


## 4. Bayes — o que a presença de laboratório indica?
Observamos que uma escola tem laboratório e queremos estimar a chance de ela atender ao critério de infraestrutura digital. Aplicamos P(C | L) = P(L | C) × P(C) / P(L).

In [5]:
p_l = L.mean()
posterior = p_l_c * p_c / p_l
print(f"P(L | C) = {p_l_c:.4f}, P(C) = {p_c:.4f}, P(L) = {p_l:.4f}")
print(f"P(C | L) = {posterior:.2%}")
print(f"Conferência: {(L & C).sum()} de {L.sum()} escolas com laboratório atendem ao critério.")

P(L | C) = 0.6336, P(C) = 0.1810, P(L) = 0.1717
P(C | L) = 66.81%
Conferência: 638 de 955 escolas com laboratório atendem ao critério.


Antes de observar o laboratório, a frequência da classe positiva é de cerca de 18%. Entre as escolas com laboratório, sobe para cerca de 67%. O uso é **diagnóstico** porque partimos de uma pista observada para inferir a classe. Uma pergunta **causal** seria quanto a instalação de um laboratório mudaria essa chance. A comparação entre grupos do censo, por si só, não mede o efeito dessa intervenção.

## 5. Laboratório e biblioteca são independentes dada a classe?
Se fossem, dentro de cada classe a frequência de escolas com ambos os recursos seria igual ao produto das duas marginais daquela classe. Basta encontrar uma combinação que viole a igualdade para descartar a independência exata na distribuição observada.

In [6]:
for classe in [0, 1]:
    grupo = C == classe
    observada = (L & B)[grupo].mean()
    produto = L[grupo].mean() * B[grupo].mean()
    print(f"C={classe} ({grupo.sum()} escolas): conjunta = {observada:.2%}, produto = {produto:.2%}")

C=0 (4555 escolas): conjunta = 5.40%, produto = 1.14%
C=1 (1007 escolas): conjunta = 49.26%, produto = 38.32%


Nas duas classes, a frequência observada dos dois recursos juntos supera o produto das marginais. Assim, laboratório e biblioteca **não são condicionalmente independentes** na distribuição empírica, mesmo separando as escolas pela classe. Isso importa para a Etapa 4 porque o Naive Bayes usa justamente essa independência como hipótese.

## Por que a lógica de primeira ordem (LPO) não expressa “70% de chance”?
Na LPO clássica, a afirmação “esta escola tem infraestrutura digital básica” é verdadeira ou falsa em uma interpretação. Uma regra universal dizendo que toda escola com laboratório tem infraestrutura digital básica seria falsa diante de um único contraexemplo. Já uma probabilidade condicional admite exceções e informa sua frequência. Para atribuir uma chance à escola sorteada precisamos definir uma distribuição, como fizemos ao dar o mesmo peso a cada escola do CSV.